01_FinTech Database Architecture, SQL Velocity Engineering & Regional Fraud EDA

In [20]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

# 1. Connect to SQLite database
conn = sqlite3.connect("fintech_fraud.db")
cursor = conn.cursor()

# Drop existing tables if re-running
cursor.executescript("""
DROP TABLE IF EXISTS users;
DROP TABLE IF EXISTS transactions;
DROP TABLE IF EXISTS compliance_policies;
""")

# 2. Create Tables
cursor.executescript("""
CREATE TABLE users (
    user_id TEXT PRIMARY KEY,
    full_name TEXT,
    home_city TEXT,
    account_age_days INTEGER,
    kyc_verified INTEGER,
    avg_monthly_spend REAL,
    registered_device_id TEXT
);

CREATE TABLE transactions (
    txn_id TEXT PRIMARY KEY,
    user_id TEXT,
    timestamp TEXT,
    amount_inr REAL,
    merchant_category TEXT,
    txn_city TEXT,
    device_id TEXT,
    payment_remark TEXT,
    is_fraud INTEGER,
    fraud_type TEXT,
    FOREIGN KEY (user_id) REFERENCES users(user_id)
);

CREATE TABLE compliance_policies (
    policy_id TEXT PRIMARY KEY,
    rule_code TEXT,
    title TEXT,
    risk_category TEXT,
    policy_text TEXT,
    mandatory_action TEXT
);
""")

# 3. Generate 200 Users with your updated locations
cities = ["Chennai", "Punjab", "Bangalore", "Mumbai", "Delhi", "Hyderabad"]
users_data = []
for i in range(1, 201):
    uid = f"USR_{i:04d}"
    city = random.choice(cities)
    age_days = random.randint(15, 1200)
    kyc = 1 if random.random() > 0.12 else 0  # 88% KYC verified
    avg_spend = round(np.random.uniform(3000, 25000), 2)
    dev_id = f"DEV_{random.randint(10000, 99999)}"
    users_data.append((uid, f"Customer_{i}", city, age_days, kyc, avg_spend, dev_id))

cursor.executemany("INSERT INTO users VALUES (?, ?, ?, ?, ?, ?, ?)", users_data)

# 4. Generate 2,000 Transactions (Normal + 3 Fraud Patterns)
categories = ["Groceries", "Utilities", "Food_Delivery", "Peer_Transfer", "Electronics", "Travel"]
normal_remarks = ["Dinner split", "Rent", "Grocery bill", "Cab fare", "Monthly recharge", "Coffee", ""]
scam_remarks = [
    "Urgent KYC block unblock fee",
    "Lottery prize tax release",
    "Customer care refund QR verify",
    "Part time job deposit bonus"
]

start_date = datetime(2026, 10, 1, 8, 0, 0)
txns_data = []

for t in range(1, 2001):
    txn_id = f"TXN_{t:05d}"
    user = random.choice(users_data)
    uid, _, home_city, age_days, kyc, avg_spend, reg_device = user

    # Random timestamp over 45 days
    ts = start_date + timedelta(
        days=random.randint(0, 44),
        hours=random.randint(7, 22),
        minutes=random.randint(0, 59)
    )

    # Default: Normal Transaction (~94% of traffic)
    if random.random() > 0.06:
        amount = round(np.random.lognormal(mean=6.2, sigma=0.8), 2)
        cat = random.choice(categories)
        city = home_city if random.random() > 0.08 else random.choice(cities)
        device = reg_device
        remark = random.choice(normal_remarks)
        is_fraud = 0
        fraud_type = "None"
    else:
        # Inject 1 of 3 Realistic Fraud Scenarios (~6% of traffic)
        pattern = random.choice(["Account_Takeover", "Velocity_Burst", "Social_Engineering_Scam"])
        is_fraud = 1
        fraud_type = pattern

        if pattern == "Account_Takeover":
            ts = ts.replace(hour=random.randint(1, 4))
            amount = round(np.random.uniform(45000, 98000), 2)
            cat = "Peer_Transfer"
            city = random.choice([c for c in cities if c != home_city])
            device = f"NEW_DEV_{random.randint(80000, 99999)}"
            remark = "Urgent transfer"

        elif pattern == "Velocity_Burst":
            amount = round(np.random.uniform(9500, 19900), 2)
            cat = "Peer_Transfer"
            city = home_city
            device = reg_device
            remark = "Transfer"

        elif pattern == "Social_Engineering_Scam":
            amount = round(np.random.uniform(5000, 35000), 2)
            cat = "Peer_Transfer"
            city = home_city
            device = reg_device
            remark = random.choice(scam_remarks)

    txns_data.append((
        txn_id, uid, ts.strftime("%Y-%m-%d %H:%M:%S"),
        amount, cat, city, device, remark, is_fraud, fraud_type
    ))

# Add rapid-fire bursts for Velocity_Burst users so SQL window functions catch them
velocity_txns = [t for t in txns_data if t[9] == "Velocity_Burst"][:15]
extra_counter = 2001
for vt in velocity_txns:
    base_time = datetime.strptime(vt[2], "%Y-%m-%d %H:%M:%S")
    for burst_idx in range(1, 5):  # 4 rapid transfers within 8 minutes
        burst_ts = (base_time + timedelta(minutes=burst_idx * 2)).strftime("%Y-%m-%d %H:%M:%S")
        txns_data.append((
            f"TXN_{extra_counter:05d}", vt[1], burst_ts,
            round(vt[3] - burst_idx * 150, 2), "Peer_Transfer",
            vt[5], vt[6], "Split transfer", 1, "Velocity_Burst"
        ))
        extra_counter += 1

cursor.executemany("INSERT INTO transactions VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)", txns_data)

# 5. Insert Compliance & Regulatory Policies (For Phase 3 RAG)
policies_data = [
    ("POL_01", "AML-101", "High-Value Nighttime & New Device Rule", "Account Takeover",
     "Any peer transfer exceeding INR 40,000 between 12:00 AM and 5:00 AM from an unregistered device or out-of-state IP must be immediately blocked and escalated for biometric re-verification.",
     "BLOCK_FREEZE"),
    ("POL_02", "AML-204", "Structuring & Rapid Velocity Threshold", "Money Laundering",
     "Accounts executing 4 or more peer transfers within a 15-minute window totaling over INR 35,000 violate anti-structuring norms and require a 24-hour cooling-off hold.",
     "HOLD_24H"),
    ("POL_03", "RBI-SCAM-09", "Social Engineering & KYC Scam Interception", "Consumer Protection",
     "Transactions containing payment remarks referencing KYC unblock fees, lottery taxes, or refund QR verification must trigger an in-app scam warning and step-up OTP challenge.",
     "CHALLENGE_2FA"),
    ("POL_04", "KYC-302", "Unverified Account Transaction Cap", "Compliance",
     "Users with incomplete KYC (kyc_verified = 0) are restricted to a maximum single transaction of INR 10,000 and cumulative monthly volume of INR 50,000.",
     "BLOCK_FREEZE"),
    ("POL_05", "SAR-500", "Suspicious Activity Report (SAR) Filing Mandate", "Regulatory Reporting",
     "Any confirmed fraud event exceeding INR 50,000 or involving multi-device account takeover requires an automated Suspicious Activity Report filed within 24 hours.",
     "FILE_SAR")
]
cursor.executemany("INSERT INTO compliance_policies VALUES (?, ?, ?, ?, ?, ?)", policies_data)

conn.commit()

# 6. Quick Verification Check
df_summary = pd.read_sql_query("""
SELECT txn_city, COUNT(*) as total_txns, SUM(is_fraud) as fraud_txns
FROM transactions
GROUP BY txn_city
ORDER BY total_txns DESC
""", conn)

print("✅ Database 'fintech_fraud.db' created with updated locations!\n")
print(df_summary)
conn.close()

✅ Database 'fintech_fraud.db' created with updated locations!

    txn_city  total_txns  fraud_txns
0     Mumbai         391          35
1  Bangalore         383          16
2  Hyderabad         378          51
3      Delhi         326          35
4    Chennai         298          15
5     Punjab         284          27


In [21]:
import sqlite3
import pandas as pd
import plotly.express as px

conn = sqlite3.connect("fintech_fraud.db")

df_viz = pd.read_sql_query("""
SELECT
    txn_city,
    fraud_type,
    COUNT(*) AS txn_count,
    ROUND(SUM(amount_inr), 2) AS total_inr
FROM transactions
WHERE is_fraud = 1
GROUP BY txn_city, fraud_type
ORDER BY total_inr DESC
""", conn)
conn.close()

fig = px.bar(
    df_viz,
    x="txn_city",
    y="total_inr",
    color="fraud_type",
    barmode="group",
    text_auto=".2s",
    category_orders={
        "txn_city": ["Bangalore", "Punjab", "Chennai", "Mumbai", "Delhi", "Hyderabad"]
    },
    title="🚨 Fraudulent Transaction Volume (INR) by City & Attack Pattern",
    labels={"txn_city": "City", "total_inr": "Fraud Volume (INR)", "fraud_type": "Attack Type"}
)
fig.update_layout(template="plotly_dark", height=450)
fig.show()

02_ Time-Series Spike Forecasting, ML Risk Scoring & Prevention Decision Engine

In [22]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# 1. Extract SQL Features (Behavioral + Velocity + NLP Keyword Flags)
conn = sqlite3.connect("fintech_fraud.db")

feature_query = """
SELECT
    t.txn_id,
    t.user_id,
    t.timestamp,
    t.amount_inr,
    t.txn_city,
    u.home_city,
    u.account_age_days,
    u.kyc_verified,
    u.avg_monthly_spend,
    t.payment_remark,
    ROUND(t.amount_inr / (u.avg_monthly_spend + 1), 2) AS spend_to_avg_ratio,
    CASE WHEN t.device_id != u.registered_device_id THEN 1 ELSE 0 END AS is_new_device,
    CASE WHEN t.txn_city != u.home_city THEN 1 ELSE 0 END AS is_city_mismatch,
    CASE WHEN CAST(strftime('%H', t.timestamp) AS INTEGER) BETWEEN 0 AND 5 THEN 1 ELSE 0 END AS is_late_night,
    COUNT(*) OVER (
        PARTITION BY t.user_id
        ORDER BY strftime('%s', t.timestamp)
        RANGE BETWEEN 900 PRECEDING AND CURRENT ROW
    ) AS txns_last_15_mins,
    CASE WHEN LOWER(t.payment_remark) LIKE '%kyc%'
           OR LOWER(t.payment_remark) LIKE '%lottery%'
           OR LOWER(t.payment_remark) LIKE '%refund%'
           OR LOWER(t.payment_remark) LIKE '%bonus%' THEN 1 ELSE 0 END AS has_scam_keyword,
    t.is_fraud,
    t.fraud_type
FROM transactions t
JOIN users u ON t.user_id = u.user_id
ORDER BY t.timestamp
"""

df = pd.read_sql_query(feature_query, conn)
df["timestamp"] = pd.to_datetime(df["timestamp"])

# 2. Time-Series Forecasting: 7-Day Fraud Volume Forecast (Rolling Trend + Weekly Seasonality)
df["date"] = df["timestamp"].dt.date
daily_fraud = df[df["is_fraud"] == 1].groupby("date")["amount_inr"].sum().reset_index()
daily_fraud["date"] = pd.to_datetime(daily_fraud["date"])
daily_fraud = daily_fraud.set_index("date").asfreq("D", fill_value=0)

dow_pattern = daily_fraud.groupby(daily_fraud.index.dayofweek)["amount_inr"].mean()
recent_baseline = daily_fraud["amount_inr"].tail(14).mean()
overall_mean = daily_fraud["amount_inr"].mean() or 1.0

future_dates = pd.date_range(daily_fraud.index[-1] + pd.Timedelta(days=1), periods=7, freq="D")
forecast_vals = [
    max(0, round(recent_baseline * (dow_pattern.get(d.dayofweek, overall_mean) / overall_mean), 2))
    for d in future_dates
]

fig_ts = go.Figure()
fig_ts.add_trace(go.Scatter(
    x=daily_fraud.index, y=daily_fraud["amount_inr"],
    mode="lines+markers", name="Historical Daily Fraud (INR)", line=dict(color="#00CC96")
))
fig_ts.add_trace(go.Scatter(
    x=future_dates, y=forecast_vals,
    mode="lines+markers", name="7-Day Forecasted Fraud (INR)",
    line=dict(color="#EF553B", dash="dash")
))
fig_ts.update_layout(
    title="📈 Time-Series Forecast: 7-Day Expected Fraud Spike Window",
    xaxis_title="Date", yaxis_title="Fraud Volume (INR)",
    template="plotly_dark", height=400
)
fig_ts.show()

# 3. Train ML Fraud Risk Scoring Model
feature_cols = [
    "amount_inr", "account_age_days", "kyc_verified",
    "spend_to_avg_ratio", "is_new_device", "is_city_mismatch",
    "is_late_night", "txns_last_15_mins", "has_scam_keyword"
]

X = df[feature_cols]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=150, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)

df["fraud_risk_score"] = clf.predict_proba(X)[:, 1].round(4)
roc_auc = roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1])

# 4. Automated Prevention Decision Engine
def prevention_decision_engine(row):
    score = row["fraud_risk_score"]
    if score >= 0.90 or (row["kyc_verified"] == 0 and row["amount_inr"] > 10000):
        return "BLOCK_FREEZE"
    elif score >= 0.75:
        return "HOLD_24H"
    elif score >= 0.40 or row["has_scam_keyword"] == 1:
        return "CHALLENGE_2FA"
    else:
        return "APPROVE"

df["prevention_action"] = df.apply(prevention_decision_engine, axis=1)

# Save scored transactions back to SQLite for Phase 3
df_to_save = df.drop(columns=["date"])
df_to_save["timestamp"] = df_to_save["timestamp"].astype(str)
df_to_save.to_sql("scored_transactions", conn, if_exists="replace", index=False)
conn.close()

# 5. Print Model Performance & Business Prevention Summary
print(f"✅ ML Fraud Model ROC-AUC Score: {roc_auc:.4f}\n")
action_summary = df.groupby("prevention_action").agg(
    txn_count=("txn_id", "count"),
    true_frauds_caught=("is_fraud", "sum"),
    total_inr_volume=("amount_inr", "sum")
).reset_index()
action_summary["total_inr_volume"] = action_summary["total_inr_volume"].round(2)

print("🛡️ Automated Prevention Engine Summary:")
print(action_summary)

✅ ML Fraud Model ROC-AUC Score: 0.9999

🛡️ Automated Prevention Engine Summary:
  prevention_action  txn_count  true_frauds_caught  total_inr_volume
0           APPROVE       1879                   0        1295470.25
1      BLOCK_FREEZE        178                 177        4585551.00
2     CHALLENGE_2FA          1                   0           6339.06
3          HOLD_24H          2                   2          16317.81


03_Gen AI Compliance Copilot(Gemini Text-to-SQL & RAG Policy Agent)

In [23]:
import sqlite3
import pandas as pd
import re
from google import genai
from google.colab import userdata

# 1. Initialize Gemini Client
api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

# 2. Safety Guardrail: Enforce Read-Only SQL
def validate_safe_sql(sql_query: str) -> str:
    cleaned = sql_query.strip().removeprefix("```sql").removeprefix("```").removesuffix("```").strip()
    forbidden = r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|REPLACE|ATTACH)\b"
    if re.search(forbidden, cleaned, re.IGNORECASE) or not cleaned.upper().startswith("SELECT"):
        raise ValueError("🚨 Security Guardrail Triggered: Only read-only SELECT queries are permitted!")
    return cleaned

# 3. Hybrid Text-to-SQL + RAG Copilot (With Instant Offline Fallback)
def ask_compliance_copilot(user_question: str):
    conn = sqlite3.connect("fintech_fraud.db")
    policies_df = pd.read_sql_query("SELECT * FROM compliance_policies", conn)
    policies_context = policies_df.to_string(index=False)

    schema_info = """
    Table: scored_transactions
    Columns: txn_id, user_id, timestamp, amount_inr, txn_city, home_city, account_age_days,
             kyc_verified, avg_monthly_spend, payment_remark, spend_to_avg_ratio,
             is_new_device, is_city_mismatch, is_late_night, txns_last_15_mins,
             has_scam_keyword, is_fraud, fraud_type, fraud_risk_score, prevention_action
    Values for prevention_action: 'BLOCK_FREEZE', 'HOLD_24H', 'CHALLENGE_2FA', 'APPROVE'
    Values for fraud_type: 'Account_Takeover', 'Velocity_Burst', 'Social_Engineering_Scam', 'None'
    """

    # Step A: Try Live Gemini Text-to-SQL using gemini-3.5-flash-lite
    try:
        sql_resp = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=f"Convert to a single valid SQLite SELECT query. Return ONLY raw SQL:\n{schema_info}\nQuestion: {user_question}"
        )
        safe_sql = validate_safe_sql(sql_resp.text)
    except Exception:
        print("⚡ Switching seamlessly to Local Deterministic SQL & RAG Engine!\n")
        safe_sql = """
        SELECT txn_id, user_id, timestamp, amount_inr, txn_city, home_city,
               kyc_verified, fraud_type, fraud_risk_score, prevention_action
        FROM scored_transactions
        WHERE prevention_action = 'BLOCK_FREEZE'
          AND txn_city IN ('Chennai', 'Bangalore')
        ORDER BY amount_inr DESC
        LIMIT 5;
        """

    print(f"🔍 Executed Safe SQL:\n{safe_sql.strip()}\n")
    query_result_df = pd.read_sql_query(safe_sql, conn)
    conn.close()

    # Step B: Try Live Gemini RAG Briefing, else Local Policy Matcher
    try:
        rag_prompt = f"""
        You are an Autonomous FinTech Fraud & Compliance Officer.
        Cite exact rule codes (AML-101, AML-204, RBI-SCAM-09, KYC-302, SAR-500) using ONLY these SQL results and policies.
        Question: {user_question}
        SQL Results:\n{query_result_df.head(10).to_string(index=False)}
        Policies:\n{policies_context}
        """
        briefing_text = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=rag_prompt
        ).text
    except Exception:
        top_row = query_result_df.iloc[0]
        briefing_text = (
            "### 🤖 Autonomous Compliance Briefing (RAG Grounded)\n"
            f"- **Top Flagged Transfer:** `{top_row['txn_id']}` (`{top_row['user_id']}`) for **₹{top_row['amount_inr']:,.2f}** in **{top_row['txn_city']}** (Home: **{top_row['home_city']}**).\n"
            "- **Triggered Rules:**\n"
            "  1. **`AML-101` (High-Value Nighttime & New Device Rule):** Transfer > ₹40,000 between 12 AM–5 AM from an unregistered device/city → Immediate `BLOCK_FREEZE` + Biometric step-up required.\n"
            "  2. **`KYC-302` (Unverified Account Cap):** Accounts with `kyc_verified = 0` are capped at ₹10,000 per transfer.\n"
            "  3. **`SAR-500` (Regulatory Reporting):** Confirmed fraud > ₹50,000 requires filing a Suspicious Activity Report within 24 hours."
        )

    print("📊 SQL Data Output:")
    print(query_result_df)
    print("\n" + briefing_text)

# Run Investigation
ask_compliance_copilot(
    "Show the top 5 blocked transactions in Chennai and Bangalore, and tell me which AML or RBI compliance rules apply."
)

🔍 Executed Safe SQL:
SELECT 
    txn_id, 
    user_id, 
    timestamp, 
    amount_inr, 
    txn_city, 
    fraud_type, 
    fraud_risk_score, 
    prevention_action,
    CASE 
        WHEN fraud_type = 'Account_Takeover' THEN 'RBI Master Directions on Digital Payment Security Controls & PMLA'
        WHEN fraud_type = 'Velocity_Burst' THEN 'RBI Guidelines on Fraud Risk Management & AML Transaction Monitoring'
        WHEN fraud_type = 'Social_Engineering_Scam' THEN 'RBI/NPCI Guidelines on Customer Protection and Fraud Prevention in Digital Payments'
        ELSE 'General AML Compliance & KYC/CFT Guidelines'
    END AS applicable_compliance_rules
FROM scored_transactions
WHERE prevention_action = 'BLOCK_FREEZE'
  AND txn_city IN ('Chennai', 'Bangalore')
ORDER BY timestamp DESC
LIMIT 5;

📊 SQL Data Output:
      txn_id   user_id            timestamp  amount_inr   txn_city  \
0  TXN_01536  USR_0113  2026-11-12 03:07:00    72952.18  Bangalore   
1  TXN_01027  USR_0194  2026-11-10 20:01:00

In [24]:
import sqlite3
import pandas as pd
from datetime import datetime

conn = sqlite3.connect("fintech_fraud.db")
# Fetch the #1 blocked transaction from your Phase 3 output
top_blocked = pd.read_sql_query("""
SELECT * FROM scored_transactions
WHERE prevention_action = 'BLOCK_FREEZE'
ORDER BY amount_inr DESC LIMIT 1
""", conn).iloc[0]
conn.close()

def simulate_biometric_challenge(txn_row, user_passed_biometric: bool):
    print("==========================================================")
    print(f"🚨 REAL-TIME BIOMETRIC STEP-UP TRIGGERED | {txn_row['txn_id']}")
    print("==========================================================")
    print(f"• Account ID       : {txn_row['user_id']} (Home: {txn_row['home_city']})")
    print(f"• Attempted Txn    : ₹{txn_row['amount_inr']:,.2f} in {txn_row['txn_city']} at {txn_row['timestamp']}")
    print(f"• Risk Score       : {txn_row['fraud_risk_score']*100:.1f}% (Rule: AML-101)")
    print(f"• Action Taken     : Out-of-band Liveness + Fingerprint prompt sent to registered device\n")

    if not user_passed_biometric:
        print("❌ BIOMETRIC VERIFICATION RESULT: FAILED (Liveness check timed out / Face mismatch)")
        print("🔒 FINAL SYSTEM DECISION       : PERMANENT BLOCK & DEVICE TOKEN REVOKED")
        print(f"💰 INR LOSS PREVENTED          : ₹{txn_row['amount_inr']:,.2f}")
        print(f"📄 COMPLIANCE AUTO-FILING      : SAR-500 Report queued at {datetime.now().strftime('%H:%M:%S')}")
    else:
        print("✅ BIOMETRIC VERIFICATION RESULT: VERIFIED (Hardware Passkey + 3D Face Match 99.4%)")
        print("🔓 FINAL SYSTEM DECISION       : HOLD LIFTED -> TRANSACTION APPROVED")
    print("==========================================================\n")

# Simulate Case A: Hacker fails biometric check on TXN_00640
simulate_biometric_challenge(top_blocked, user_passed_biometric=False)

🚨 REAL-TIME BIOMETRIC STEP-UP TRIGGERED | TXN_00809
• Account ID       : USR_0198 (Home: Mumbai)
• Attempted Txn    : ₹96,314.25 in Delhi at 2026-11-10 02:17:00
• Risk Score       : 100.0% (Rule: AML-101)
• Action Taken     : Out-of-band Liveness + Fingerprint prompt sent to registered device

❌ BIOMETRIC VERIFICATION RESULT: FAILED (Liveness check timed out / Face mismatch)
🔒 FINAL SYSTEM DECISION       : PERMANENT BLOCK & DEVICE TOKEN REVOKED
💰 INR LOSS PREVENTED          : ₹96,314.25
📄 COMPLIANCE AUTO-FILING      : SAR-500 Report queued at 16:08:11



In [25]:
import sqlite3
import pandas as pd
from datetime import datetime

conn = sqlite3.connect("fintech_fraud.db")
# Fetch TXN_00640 specifically
top_blocked = pd.read_sql_query("""
SELECT * FROM scored_transactions
WHERE txn_id = 'TXN_00640'
""", conn).iloc[0]
conn.close()

def simulate_biometric_challenge(txn_row, user_passed_biometric: bool):
    print("==========================================================")
    print(f"🚨 REAL-TIME BIOMETRIC STEP-UP TRIGGERED | {txn_row['txn_id']}")
    print("==========================================================")
    print(f"• Account ID       : {txn_row['user_id']} (Home: {txn_row['home_city']})")
    print(f"• Attempted Txn    : ₹{txn_row['amount_inr']:,.2f} in {txn_row['txn_city']} at {txn_row['timestamp']}")
    print(f"• Risk Score       : {txn_row['fraud_risk_score']*100:.1f}% (Rule: AML-101)")
    print(f"• Action Taken     : Out-of-band Liveness + Fingerprint prompt sent to registered device\n")

    if not user_passed_biometric:
        print("❌ BIOMETRIC VERIFICATION RESULT: FAILED (Liveness check timed out / Face mismatch)")
        print("🔒 FINAL SYSTEM DECISION       : PERMANENT BLOCK & DEVICE TOKEN REVOKED")
        print(f"💰 INR LOSS PREVENTED          : ₹{txn_row['amount_inr']:,.2f}")
        print(f"📄 COMPLIANCE AUTO-FILING      : SAR-500 Report queued at {datetime.now().strftime('%H:%M:%S')}")
    else:
        print("✅ BIOMETRIC VERIFICATION RESULT: VERIFIED (Hardware Passkey + 3D Face Match 99.4%)")
        print("🔓 FINAL SYSTEM DECISION       : HOLD LIFTED -> TRANSACTION APPROVED")
    print("==========================================================\n")

# Simulate Case A: Hacker fails biometric check on TXN_00640
simulate_biometric_challenge(top_blocked, user_passed_biometric=False)

🚨 REAL-TIME BIOMETRIC STEP-UP TRIGGERED | TXN_00640
• Account ID       : USR_0186 (Home: Bangalore)
• Attempted Txn    : ₹93,520.88 in Chennai at 2026-10-04 03:38:00
• Risk Score       : 100.0% (Rule: AML-101)
• Action Taken     : Out-of-band Liveness + Fingerprint prompt sent to registered device

❌ BIOMETRIC VERIFICATION RESULT: FAILED (Liveness check timed out / Face mismatch)
🔒 FINAL SYSTEM DECISION       : PERMANENT BLOCK & DEVICE TOKEN REVOKED
💰 INR LOSS PREVENTED          : ₹93,520.88
📄 COMPLIANCE AUTO-FILING      : SAR-500 Report queued at 16:08:11



04_Interactive Streamlit Dashboard & Gemini Voice Assistant

In [26]:
!pip install -q streamlit gTTS

In [31]:
%%writefile app.py
import streamlit as st
import sqlite3
import pandas as pd
import plotly.express as px
import re
import io
from gtts import gTTS
from google import genai
from google.genai import types

st.set_page_config(page_title="UPI Guardian: FinTech Fraud & Compliance Copilot", page_icon="🛡️", layout="wide")

st.title("🛡️ UPI Guardian: Autonomous FinTech Fraud & Compliance Copilot")
st.caption("Real-Time Anomaly Prevention • Time-Series Spike Forecasting • Gemini Text-to-SQL & RAG • Hands-Free Voice Assistant")

# Sidebar: API Key & Filters
with st.sidebar:
    st.header("🔑 Gemini AI Configuration")
    api_key = st.text_input("Enter Gemini API Key (Paste AIza... for Live Voice/LLM)", type="password")
    st.divider()
    st.header("📍 Regional Filter")
    selected_cities = st.multiselect(
        "Select Cities",
        ["Bangalore", "Punjab", "Chennai", "Mumbai", "Delhi", "Hyderabad"],
        default=["Bangalore", "Punjab", "Chennai", "Mumbai", "Delhi", "Hyderabad"]
    )

@st.cache_data
def load_data():
    conn = sqlite3.connect("fintech_fraud.db")
    df = pd.read_sql_query("SELECT * FROM scored_transactions", conn)
    policies = pd.read_sql_query("SELECT * FROM compliance_policies", conn)
    conn.close()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df, policies

df, policies_df = load_data()
filtered_df = df[df["txn_city"].isin(selected_cities)]

# Top KPI Row
col1, col2, col3, col4 = st.columns(4)
blocked_df = filtered_df[filtered_df["prevention_action"].isin(["BLOCK_FREEZE", "HOLD_24H"])]
col1.metric("Total Transactions", f"{len(filtered_df):,}")
col2.metric("Fraudulent Attempts", f"{int(filtered_df['is_fraud'].sum()):,}")
col3.metric("Blocked / Held Transfers", f"{len(blocked_df):,}")
col4.metric("INR Losses Prevented", f"₹{blocked_df[blocked_df['is_fraud']==1]['amount_inr'].sum():,.0f}")

st.divider()

tab1, tab2, tab3 = st.tabs(["📊 Fraud Analytics & Prevention", "🤖 AI Compliance Copilot (Text + Voice)", "📜 Regulatory Policies & Live Ledger"])

with tab1:
    c1, c2 = st.columns(2)
    with c1:
        fraud_city = filtered_df[filtered_df["is_fraud"] == 1].groupby(["txn_city", "fraud_type"])["amount_inr"].sum().reset_index()
        fig_bar = px.bar(
            fraud_city, x="txn_city", y="amount_inr", color="fraud_type", barmode="group",
            category_orders={"txn_city": ["Bangalore", "Punjab", "Chennai", "Mumbai", "Delhi", "Hyderabad"]},
            title="🚨 Fraud Volume (INR) by City & Attack Pattern", template="plotly_dark"
        )
        st.plotly_chart(fig_bar, use_container_width=True)
    with col2:
        action_counts = filtered_df["prevention_action"].value_counts().reset_index()
        action_counts.columns = ["Action", "Count"]
        fig_pie = px.pie(
            action_counts, names="Action", values="Count", hole=0.45,
            title="🛡️ Real-Time Prevention Decision Split", template="plotly_dark"
        )
        st.plotly_chart(fig_pie, use_container_width=True)

with tab2:
    st.subheader("🎙️ Ask the FinTech Compliance Copilot (Voice or Text)")
    audio_value = st.audio_input("Speak your investigation query (e.g., 'Find velocity burst transactions in Hyderabad')")
    text_query = st.text_input("Or type your question here:", value="Show the top 5 blocked transactions in Chennai and Bangalore, and tell me which AML or RBI compliance rules apply.")

    if st.button("🚀 Run Investigation", type="primary"):
        final_question = text_query
        client = genai.Client(api_key=api_key) if api_key else None

        # 1. Live Voice Transcription if Audio Provided + API Key Present
        if audio_value is not None:
            if client is not None:
                try:
                    with st.spinner("🎧 Transcribing your voice query with Gemini..."):
                        audio_bytes = audio_value.read()
                        transcribe_resp = client.models.generate_content(
                            model="gemini-3.5-flash-lite",
                            contents=[
                                "Transcribe this spoken FinTech fraud investigation question accurately. Return ONLY the text.",
                                types.Part.from_bytes(data=audio_bytes, mime_type="audio/wav")
                            ]
                        )
                        final_question = transcribe_resp.text.strip()
                        st.info(f"🗣️ **Transcribed Voice Query:** {final_question}")
                except Exception:
                    st.warning("⚡ Live voice API quota busy — using your text query box.")
            else:
                st.warning("🔑 Paste your Gemini API Key (`AIza...`) in the left sidebar for live voice transcription—using your text query box for now.")

        if final_question:
            with st.spinner("🔍 Generating Safe SQL & Checking RBI/AML Compliance Rules..."):
                schema_info = """
                Table: scored_transactions
                Columns: txn_id, user_id, timestamp, amount_inr, txn_city, home_city, account_age_days,
                         kyc_verified, avg_monthly_spend, payment_remark, spend_to_avg_ratio,
                         is_new_device, is_city_mismatch, is_late_night, txns_last_15_mins,
                         has_scam_keyword, is_fraud, fraud_type, fraud_risk_score, prevention_action
                Values for prevention_action: 'BLOCK_FREEZE', 'HOLD_24H', 'CHALLENGE_2FA', 'APPROVE'
                Values for fraud_type: 'Account_Takeover', 'Velocity_Burst', 'Social_Engineering_Scam', 'None'
                """
                # 2. Text-to-SQL (Live Gemini with Smart Keyword-Aware Fallback)
                try:
                    if client is None:
                        raise ValueError("No API Key")
                    sql_resp = client.models.generate_content(
                        model="gemini-3.5-flash-lite",
                        contents=f"Convert into a single valid SQLite SELECT query. Return ONLY raw SQL:\n{schema_info}\nQuestion: {final_question}"
                    )
                    cleaned_sql = sql_resp.text.strip().removeprefix("```sql").removeprefix("```").removesuffix("```").strip()
                except Exception:
                    st.info("⚡ Running Smart Deterministic SQL & RAG Compliance Engine")
                    q_lower = final_question.lower()
                    cities = [c for c in ["Chennai", "Bangalore", "Hyderabad", "Mumbai", "Delhi", "Punjab"] if c.lower() in q_lower]
                    city_clause = f"AND txn_city IN ({','.join([repr(c) for c in cities])})" if cities else ""

                    if "velocity" in q_lower:
                        type_clause = "AND fraud_type = 'Velocity_Burst'"
                    elif "takeover" in q_lower:
                        type_clause = "AND fraud_type = 'Account_Takeover'"
                    elif "scam" in q_lower:
                        type_clause = "AND fraud_type = 'Social_Engineering_Scam'"
                    elif "kyc" in q_lower:
                        type_clause = "AND kyc_verified = 0"
                    else:
                        type_clause = "AND prevention_action = 'BLOCK_FREEZE'"

                    cleaned_sql = f"""
                    SELECT txn_id, user_id, timestamp, amount_inr, txn_city, home_city,
                           kyc_verified, fraud_type, fraud_risk_score, prevention_action
                    FROM scored_transactions
                    WHERE is_fraud = 1 {city_clause} {type_clause}
                    ORDER BY amount_inr DESC
                    LIMIT 5;
                    """

                cleaned_sql = cleaned_sql.strip()
                if re.search(r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE)\b", cleaned_sql, re.IGNORECASE) or not cleaned_sql.upper().startswith("SELECT"):
                    st.error("🚨 Security Guardrail Blocked Unsafe SQL!")
                else:
                    conn = sqlite3.connect("fintech_fraud.db")
                    res_df = pd.read_sql_query(cleaned_sql, conn)
                    conn.close()

                    st.code(cleaned_sql, language="sql")
                    st.dataframe(res_df.head(15), use_container_width=True)

                    if not res_df.empty:
                        # 3. RAG Compliance Briefing (Live Gemini with Dynamic Rule Fallback)
                        try:
                            if client is None:
                                raise ValueError("No API Key")
                            rag_prompt = f"""
                            You are an Autonomous FinTech Compliance Officer. Cite exact rule codes (AML-101, AML-204, RBI-SCAM-09, KYC-302, SAR-500).
                            Question: {final_question}
                            SQL Data: {res_df.head(10).to_string(index=False)}
                            Policies: {policies_df.to_string(index=False)}
                            """
                            briefing = client.models.generate_content(model="gemini-3.5-flash-lite", contents=rag_prompt).text
                            except Exception:
                            top_row = res_df.iloc[0]
                            q_lower = final_question.lower()
                            if "velocity" in q_lower:
                                rule_detail = "* **AML-204 (Rapid Velocity Burst Rule):** 3+ transactions within 15 minutes triggered automated velocity throttling and `HOLD_24H` / `BLOCK_FREEZE` review.\n"
                            elif "scam" in q_lower:
                                rule_detail = "* **RBI-SCAM-09 (Social Engineering Protection):** Suspicious payment remarks (`KYC`, `Lottery`, `Refund`) triggered mandatory `CHALLENGE_2FA` / `BLOCK_FREEZE`.\n"
                            elif "kyc" in q_lower:
                                rule_detail = "* **KYC-302 (Unverified Account Cap):** Accounts with `kyc_verified = 0` exceeding ₹10,000 are automatically blocked.\n"
                            else:
                                rule_detail = "* **AML-101 (High-Value Nighttime & New Device Rule):** Transfer > ₹40,000 from an unregistered device/city triggered an immediate `BLOCK_FREEZE` and Biometric step-up.\n"

                            user_str = top_row.get("user_id", "Flagged Account")
                            txn_str = f" (`{top_row['txn_id']}`)" if "txn_id" in top_row else ""
                            amt_val = top_row.get("amount_inr", top_row.get("total_fraud_amount", 0))
                            city_str = f" in **{top_row['txn_city']}**" if "txn_city" in top_row else ""

                            briefing = (
                                f"**Top Flagged Entity:** `{user_str}`{txn_str} with **₹{amt_val:,.2f}**{city_str}.\n\n"
                                f"{rule_detail}"
                                "* **SAR-500:** Confirmed fraud > ₹50,000 requires filing a Suspicious Activity Report within 24 hours."
                            )


Overwriting app.py


In [33]:
with open("app.py", "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "except Exception:" in line and i > 150:
        # Match exact indentation of the 'try:' block above it
        for j in range(i - 1, 140, -1):
            if lines[j].strip() == "try:":
                indent = lines[j][:len(lines[j]) - len(lines[j].lstrip())]
                lines[i] = indent + "except Exception:\n"
                break

with open("app.py", "w") as f:
    f.writelines(lines)

print("✅ Auto-aligned except Exception: to match try:!")
!streamlit run app.py & npx --yes localtunnel --port 8501


✅ Auto-aligned except Exception: to match try:!
⠙⠹⠸

⠼⠴⠦⠧⠇your url is: https://tasty-bears-grab.loca.lt
2026-11-26 16:52:41.606 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.66.133.188:8501

2026-11-26 16:57:43.122 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-11-26 16:57:43.175 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-11-26 16:58:23.512 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width

In [32]:
with open("app.py", "r") as f:
    code = f.read()

code = code.replace("briefing[:450]", "briefing")

with open("app.py", "w") as f:
    f.write(code)

print("✅ Updated audio readout to speak the full briefing!")
!streamlit run app.py & npx --yes localtunnel --port 8501


✅ Updated audio readout to speak the full briefing!
⠙⠹⠸

⠼⠴⠦your url is: https://good-ties-rest.loca.lt
2026-11-26 16:22:17.270 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.66.133.188:8501

2026-11-26 16:22:50.486 Script compilation error
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/streamlit/runtime/scriptrunner/script_runner.py", line 754, in _run_script
    code = self._script_cache.get_bytecode(script_path)
  File "/usr/local/lib/python3.13/dist-packages/streamlit/runtime/scriptrunner/script_cache.py", line 72, in get_bytecode
    filebody = magic.add_magic(filebody, script_path)
  File "/usr/local/lib/python3.13/dist-packages/streamlit/runtime/scriptrunner/magic.py", line 44, in add_magic
    tree = ast.parse(code, script_path, "exec")
  File "/usr/lib/python3.13/ast.py", line 50, in parse
    retur